### Data Profiling — E-commerce Sales Dataset

Quick automated overview to spot missing values, duplicates, inconsistent categories, and outliers **before** deciding which Excel formulas/fixes to apply.

### 0. Setup — install & import

In [11]:
import pandas as pd
from ydata_profiling import ProfileReport
pd.set_option('display.max_columns', None)

### 1. Load  data

In [12]:
RAW_FILE_PATH = "./data/sales_data(raw data).csv"
CLEANED_FILE_PATH = "./data/sales_data(cleaned data).csv"

raw = pd.read_csv(RAW_FILE_PATH)
cleaned = pd.read_csv(CLEANED_FILE_PATH)

cleaned['Age'] = pd.to_numeric(cleaned['Age'], errors='coerce').astype('Int64')

print("Raw DataFrame:")
display(raw.head())
print("\nCleaned DataFrame:")
display(cleaned.head())

Raw DataFrame:


,Customer ID,Gender,Region,Age,Product Name,Category,Unit Price,Quantity,Total Price,Order Date,Sales Channel,Unnamed: 11
0,CUST0268,Male,North,NaN,Monitor,Electronics,300.0,5,1500,12/8/2023,Online,NaN
1,CUST0046,Male,West,22.0,Headphones,Accessories,100.0,2,200,4/9/2023,Online,NaN
2,CUST0169,Female,South,54.0,Monitor,Electronics,300.0,1,300,8/28/2023,Marketplace,NaN
3,CUST0002,Male,North,23.0,Headphones,Accessories,100.0,5,500,1/18/2023,Retail Store,NaN
4,CUST0173,Female,South,NaN,Laptop,Electronics,1500.0,3,4500,1/19/2023,Retail Store,NaN



Cleaned DataFrame:


,Order ID,Customer ID,Gender,Division,Age,Sales Channel,Product Name,Category,Unit Price,Quantity,Total Price,Order Date,Week Days,Day Type,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,ORD-00001,CUST0268,Male,Dhaka,21,Online,Monitor,Electronics,36900.0,5.0,184500.0,12/8/2023,Friday,Weekend,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ORD-00002,CUST0046,Female,Dhaka,22,Retail Store,Headphones,Accessories,12300.0,2.0,24600.0,4/9/2023,Sunday,Weekday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD-00003,CUST0169,Female,Chattogram,54,Online,Monitor,Electronics,36900.0,1.0,36900.0,8/28/2023,Monday,Weekday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ORD-00004,CUST0002,Male,Rajshahi,23,Online,Headphones,Accessories,12300.0,5.0,61500.0,1/18/2023,Wednesday,Weekday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ORD-00005,CUST0173,Male,Dhaka,65,Marketplace,Laptop,Electronics,184500.0,3.0,553500.0,1/19/2023,Thursday,Weekday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2. Conflicting values per customer

In [13]:
# Customers with more than one distinct Gender value
gender_conflicts_raw = raw.groupby('Customer ID')['Gender'].nunique()
gender_conflicts_raw = gender_conflicts_raw[gender_conflicts_raw > 1]
print(f"Customers with conflicting Gender (Raw Data): {len(gender_conflicts_raw)}")
display(gender_conflicts_raw)

gender_conflicts_cleaned = cleaned.groupby('Customer ID')['Gender'].nunique()
gender_conflicts_cleaned = gender_conflicts_cleaned[gender_conflicts_cleaned > 1]
print(f"Customers with conflicting Gender (Cleaned): {len(gender_conflicts_cleaned)}")
display(gender_conflicts_cleaned)

Customers with conflicting Gender (Raw Data): 192


Customer ID
CUST0001    2
CUST0002    2
CUST0004    2
CUST0006    2
CUST0007    2
           ..
CUST0295    2
CUST0296    2
CUST0298    2
CUST0299    2
CUST0300    2
Name: Gender, Length: 192, dtype: int64

Customers with conflicting Gender (Cleaned): 0


Series([], Name: Gender, dtype: int64)

In [14]:
# Same check for Region
region_conflicts_raw = raw.groupby('Customer ID')['Region'].nunique() 
region_conflicts_raw = region_conflicts_raw[region_conflicts_raw > 1]
print(f"Customers with conflicting Region (Raw Data): {len(region_conflicts_raw)}")
display(region_conflicts_raw)

division_conflicts_cleaned = cleaned.groupby('Customer ID')['Division'].nunique()
division_conflicts_cleaned = division_conflicts_cleaned[division_conflicts_cleaned > 1]
print(f"Customers with conflicting Division (Cleaned): {len(division_conflicts_cleaned)}")
display(division_conflicts_cleaned)

Customers with conflicting Region (Raw Data): 217


Customer ID
CUST0002    3
CUST0003    2
CUST0004    2
CUST0006    4
CUST0007    2
           ..
CUST0295    3
CUST0296    4
CUST0297    2
CUST0298    3
CUST0299    4
Name: Region, Length: 217, dtype: int64

Customers with conflicting Division (Cleaned): 0


Series([], Name: Division, dtype: int64)

### 3. Accuracy check — does Unit Price × Quantity = Total Price?

In [15]:
expected_total_raw = (raw['Unit Price'] * raw['Quantity']).round(2)
mismatches = raw[expected_total_raw != raw['Total Price']]
print(f"Rows where Unit Price x Quantity != Total Price (Raw Data): {len(mismatches)}")
display(mismatches[['Customer ID', 'Unit Price', 'Quantity', 'Total Price']])

expected_total_cleaned = (cleaned['Unit Price'] * cleaned['Quantity']).round(2)
mismatches_cleaned = cleaned[expected_total_cleaned != cleaned['Total Price']]
print(f"Rows where Unit Price x Quantity != Total Price (Cleaned Data): {len(mismatches_cleaned)}")
display(mismatches_cleaned[['Customer ID', 'Unit Price', 'Quantity', 'Total Price']])

Rows where Unit Price x Quantity != Total Price (Raw Data): 20


,Customer ID,Unit Price,Quantity,Total Price
26,CUST0264,1658.432327,2,1600
83,CUST0219,62.191212,2,60
84,CUST0037,103.652020,4,200
110,CUST0223,62.191212,2,60
194,CUST0269,621.912123,5,1500
209,CUST0134,621.912123,5,1500
215,CUST0194,62.191212,1,30
226,CUST0264,103.652020,4,200
244,CUST0128,621.912123,3,900
339,CUST0219,207.304041,1,100


Rows where Unit Price x Quantity != Total Price (Cleaned Data): 1


,Customer ID,Unit Price,Quantity,Total Price
1956,NaN,NaN,NaN,NaN


### 4. Consistency check — does each product's price is same?

In [16]:
price_check_raw = raw.groupby('Product Name')['Unit Price'].nunique()
inconsistent_raw = price_check_raw[price_check_raw > 1]
print("Products with inconsistent pricing (Raw):", len(inconsistent_raw))
print(inconsistent_raw)
print(raw.groupby('Product Name')['Unit Price'].unique())

price_check_cleaned = cleaned.groupby('Product Name')['Unit Price'].nunique()
inconsistent_cleaned = price_check_cleaned[price_check_cleaned > 1]
print("Products with inconsistent pricing (Cleaned):", len(inconsistent_cleaned))
print(inconsistent_cleaned)
print(cleaned.groupby('Product Name')['Unit Price'].unique())

Products with inconsistent pricing (Raw): 6
Product Name
Headphones    2
Keyboard      2
Laptop        2
Monitor       2
Mouse         2
Smartphone    2
Name: Unit Price, dtype: int64
Product Name
Headphones     [100.0, 207.3040408]
Keyboard        [50.0, 103.6520204]
Laptop        [1500.0, 3109.560612]
Monitor        [300.0, 621.9121225]
Mouse           [30.0, 62.19121225]
Smartphone     [800.0, 1658.432327]
Smartwatch                  [200.0]
Name: Unit Price, dtype: object
Products with inconsistent pricing (Cleaned): 0
Series([], Name: Unit Price, dtype: int64)
Product Name
Headphones     [12300.0]
Keyboard        [6150.0]
Laptop        [184500.0]
Monitor        [36900.0]
Mouse           [3690.0]
Smartphone     [98400.0]
Smartwatch     [24600.0]
Name: Unit Price, dtype: object


### 4. Full automated report 

Generates one interactive HTML file covering nulls, duplicates, correlations, distributions, and cardinality warnings for the *entire* dataset.

In [ ]:
profile = ProfileReport(cleaned, title="E-commerce Sales Data Profile", explorative=True)
profile.to_file("data_profile_report.html")
print("Saved: data_profile_report.html — open this file in your browser")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 70.27it/s]

Saved: data_profile_report.html — open this file in your browser
